# 05 · Synthetic Data Validation
Generate synthetic MMM data with known ground-truth parameters, fit the model, and check that recovered ROIs match the true values.

In [ ]:
!pip install git+https://github.com/google/meridian.git altair -q

In [ ]:
import sys
sys.path.insert(0, '/content')
import numpy as np, pandas as pd
from src.synthetic_data import generate_synthetic_mmm_data
from src.data_loader import detect_columns, build_channel_mappings, build_meridian_dataset
from src.validation import check_data_quality

## Generate Synthetic Dataset

In [ ]:
df = generate_synthetic_mmm_data(n_weeks=104, seed=42)
df.to_csv('data/synthetic_mmm_data.csv', index=False)
print(df.shape)
df.head()

## Validate Data Quality

In [ ]:
media, impressions, output = detect_columns(df)
check_data_quality(df)
print('\nMedia:', media)
print('Impressions:', impressions)
print('KPI:', output)

## Fit Model on Synthetic Data

In [ ]:
import tensorflow_probability as tfp, tensorflow as tf
from meridian import constants
from meridian.model import model, spec, prior_distribution

cost_mapping, impressions_mapping = build_channel_mappings(media, impressions)
data_meridian = build_meridian_dataset(df, media, impressions, output, cost_mapping, impressions_mapping)

def estimate_lognormal_dist(mean, std):
    mu_log = np.log(mean) - 0.5 * np.log((std / mean) ** 2 + 1)
    std_log = np.sqrt(np.log((std / mean) ** 2 + 1))
    return mu_log, std_log

roi_mu_log, roi_sigma_log = estimate_lognormal_dist(2.0, 1.5)
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=np.float32(roi_mu_log), scale=np.float32(roi_sigma_log), name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, max_lag=7)
mmm = model.Meridian(input_data=data_meridian, model_spec=model_spec)
mmm.sample_prior(50)
mmm.sample_posterior(n_chains=3, n_adapt=500, n_burnin=500, n_keep=1000)

## Compare Recovered vs True ROI

In [ ]:
from meridian.analysis import visualizer
import matplotlib.pyplot as plt

media_summary = visualizer.MediaSummary(mmm)
media_summary.summary_table()

In [ ]:
# True ROI implied by data-generating process:
# Meta: coeff 0.8 / avg weekly spend; Google: coeff 1.2 / avg weekly spend
true_roi = {
    'Meta':   0.8  / df['Meta_spend'].mean(),
    'Google': 1.2 / df['Google_spend'].mean(),
}
print('True (implied) ROI:')
for ch, val in true_roi.items():
    print(f'  {ch}: {val:.4f}')